# XGBoost to predict discharge disability

## Import libraries

In [7]:
# Turn warnings off to keep notebook tidy
import warnings
warnings.filterwarnings("ignore")

import os
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import math
import numpy as np
import pandas as pd
import pickle
import shap

from dataclasses import dataclass
from os.path import exists
from sklearn.metrics import auc
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

## General settings

In [8]:
# K folds in data
n_kfold = 5
# Disability thresholds for binary classification models
list_binary_thresholds = [0, 1, 2, 3, 4, 5]
n_binary_models = len(list_binary_thresholds)
# Surrogate time for no thrombolysis (for SHAP dependence plots)
surrogate_time_for_no_thrombolysis = 9999

File paths

In [9]:
@dataclass(frozen=True)
class Paths:
    '''Singleton object for storing paths to data and database.'''
    image_save_path: str = './saved_images'
    model_save_path: str = './saved_models'
    data_save_path: str = './saved_data'
    data_read_path: str = './data_processing/output/kfold_5fold'
    model_text: str = (f'xgb_{n_kfold}fold_binary_model')
    notebook: str = '02_' # Prefix to file name

paths = Paths()

Create output folders if needed

In [10]:
path = paths.image_save_path
if not os.path.exists(path):
    os.makedirs(path)
        
path = paths.model_save_path
if not os.path.exists(path):
    os.makedirs(path)

path = paths.data_save_path
if not os.path.exists(path):
    os.makedirs(path)

## Features to use

In [11]:
# Features to use
selected_features = ["prior_disability", "stroke_severity", "stroke_team", 
                     "age", "onset_to_thrombolysis_time", "any_afib_diagnosis", 
                     "precise_onset_known"]

# Feature name dictionary for plotting
dict_feature_names = {}
dict_feature_names["prior_disability"] = "Prior disability (mRS)"
dict_feature_names["stroke_severity"] = "Stroke severity (NIHSS)"
dict_feature_names["stroke_team"] = "Stroke team"
dict_feature_names["age"] = "Age (years)"
dict_feature_names["onset_to_thrombolysis_time"] = "Onset to thrombolysis time (minutes)"
dict_feature_names["any_afib_diagnosis"] = "Atrial fibrilation diagnosis"
dict_feature_names["precise_onset_known"] = "Precise onset time known"
dict_feature_names['discharge_disability'] = "Discharge disabiliity (mRS)"

# Number of features
n_features = len(selected_features)

# Target feature
target_feature = 'discharge_disability'

## Load data

In [6]:
X_train_kfold, y_train_kfold, X_test_kfold, y_test_kfold = [], [], [], []

for k in range(n_kfold):
    # Read in training set, restrict to chosen features & store
    filename = os.path.join(paths.data_read_path, 
                            ('train_' + str(k) + '.csv'))
    train = pd.read_csv(filename)
    X_train = train[selected_features]
    y_train = train[target_feature]
    X_train_kfold.append(X_train)
    y_train_kfold.append(y_train)

    # Read in test set, restrict to chosen features & store
    filename = os.path.join(paths.data_read_path, 
                            ('test_' + str(k) + '.csv'))
    test = pd.read_csv(filename)
    X_test = test[selected_features]
    y_test = test[target_feature]
    X_test_kfold.append(X_test)
    y_test_kfold.append(y_test)